# GAN (生成的敵対ネットワーク）

### prompt
```prompt
GANとは？
```
### answer
gpt4oの回答

#### 基本構造：Generator と Discriminator

GANは2つのニューラルネットワークから構成されます：

---

##### 1. Generator（生成器）

* ランダムなノイズ（例えば100次元の乱数ベクトル）を入力として受け取り、
* 本物のようなデータ（画像など）を**生成**します。
* 目標は、生成データが本物に見えるように Discriminator を騙すこと。

---

##### 2. Discriminator（識別器）

* 入力されたデータが「本物」か「Generator による偽物」かを分類します。
* 本物のデータ（例：本当の画像）と、Generator が作った偽データの両方を受け取り、判定します。

この2つは**ゲームのように競い合いながら**訓練されます。

#### 🎯 学習の仕組み（ミニマックスゲーム）

学習の流れは以下の通りです：

1. Generator がノイズからデータを生成
2. Discriminator は、それが本物か偽物かを判定
3. Generator は「Discriminator を騙すように」学習を進める
4. Discriminator は「偽物を見抜くように」学習を進める

つまり、以下のような **ゼロサムゲーム** が行われます：

* Generator の目的：Discriminator を騙す
* Discriminator の目的：Generator の偽物を見抜く

この対立により、Generator はよりリアルなデータを作れるようになります。

### prompt
```prompt
ミニマックスゲーム、ゼロサムゲームとは？
```

### answer

#### ✅ ミニマックスゲームとは

ミニマックス（Minimax）ゲームは、主に **2人対戦のゼロサムゲーム** において用いられる戦略で、
**自分にとって最悪の状況を最小限に抑える（= Minimize the Maximum loss）** ことを目的とした意思決定の方法です。

---

#### ✅ ゼロサムゲームとは

ゼロサム（Zero-sum）ゲームとは、参加者全体の利得（得点・利益）の合計が常に「ゼロ」になるゲームです。

***(補足：どちらかが一方的に勝ってはいけない。）***

---

#### 🧠 ミニマックスとゼロサムの関係

* ミニマックス戦略はゼロサムゲームにおける**最適戦略**として理論化されており、対戦型ゲームAIでも基礎技術になっています。
* フォン・ノイマンが提唱したミニマックス定理によって、ゼロサムゲームには常にミニマックス戦略による**均衡点（ナッシュ均衡）**が存在することが示されました。


天下りですが、

## GANのnetwork
![](fig/8200-gan-diagram1.jpg)

---

## GANで学習するモデルと損失関数
![](fig/8200-gan-diagram2.jpg)


## スクリプト生成

ある程度のvariationのある物性値画像のデータが無いので、手書き文字を再び用いています。

### prompt
```prompt
gpt4oに
Scikit-learnの手書き文字に対して、画像を[-1,1]で規格化し
Generatorは[-1,1]を出力し
Discriminatorは[0,1]を出力する。
BCE lossを用いて、
Fully connected neural networkモデルでGAN作成してください。
```

### answer
出力コードに対して、
出力の値の範囲を確認、次元を変数化などの変更したのが以下のコードです。


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, confusion_matrix
import numpy as np

# Training parameters
EPOCHS = 200
DISPLAY_PER_EPOCHS = 20
BATCH_SIZE = 32

LATENT_DIM = 64
HIDDEN_DIM_A = 128
HIDDEN_DIM_B = 256

IMG_DIM = 64 # (8,8)の画像なので64に固定

# Load dataset
digits = load_digits()
X = digits.data / digits.data.max() *2-1 # Normalize pixel values between 0 and 1
y = digits.target

# Split dataset into training and test sets (80% train, 20% test)
X_train, X_test, _, _ = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)

# Generator model
class Generator(nn.Module):
    def __init__(self, latent_dim, hidden_dim1, hidden_dim2, img_dim):
        super(Generator, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim1),
            # nn.BatchNorm1d(hidden_dim1), 
            
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            # nn.BatchNorm1d(hidden_dim2), 
            
            nn.ReLU(),
            nn.Linear(hidden_dim2, img_dim),  # Output size matches the image size (img_dim = 8x8 = 64)
            nn.Tanh()  # Assuming image pixel values are normalized between -1 and 1
        )

    def forward(self, x):
        return self.fc(x)
    


class Discriminator(nn.Module):
    def __init__(self, img_dim, hidden_dim1, hidden_dim2):
        super(Discriminator, self).__init__()
        self.fc = nn.Sequential(
            nn.Linear(img_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim2, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.fc(x)

# Initialize models
generator = Generator(latent_dim = LATENT_DIM, img_dim = IMG_DIM, hidden_dim1=HIDDEN_DIM_A, hidden_dim2=HIDDEN_DIM_B)
discriminator = Discriminator(img_dim = IMG_DIM, hidden_dim1=HIDDEN_DIM_B, hidden_dim2=HIDDEN_DIM_A)

# Loss function and optimizers
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=0.001)
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.001)

# For plotting losses
losses_G = []
losses_D = []

# Training GAN
for epoch in range(EPOCHS):
    batch_G_loss = []
    batch_D_loss = []    
    for i in range(0, X_train.size(0), BATCH_SIZE):
        real_data = X_train[i:i + BATCH_SIZE]
        batch_size_real = real_data.size(0)
        
        # Labels for real and fake data
        valid = torch.ones(batch_size_real, 1)   # (バッチサイズ, 1)の配列。値は１
        fake = torch.zeros(batch_size_real, 1)  # (バッチサイズ, 1)の配列。値は0

        # Train Discriminator
        optimizer_D.zero_grad()

        # Generate fake data, noiseが乱数なので毎回生成imageは異なる。
        noise = torch.randn(batch_size_real, LATENT_DIM) # (バッチサイズ, LATENT_DIM)の配列,値は正規分布
        fake_data = generator(noise)

        # Discriminator loss on real and fake data
        real_loss = criterion(discriminator(real_data), valid) # real_dataはvalidに
        fake_loss = criterion(discriminator(fake_data.detach()), fake) # fake_dataはfakeとなるように最適化する。
        loss_D = real_loss + fake_loss
        batch_D_loss.append(loss_D.item())
        loss_D.backward()
        optimizer_D.step()

        # Train Generator
        optimizer_G.zero_grad()

        # Generator loss (Discriminator should predict that the fake data is real)
        loss_G = criterion(discriminator(fake_data), valid)
        batch_G_loss.append(loss_G.item())
        
        loss_G.backward()
        optimizer_G.step()

    losses_G.append(np.array(batch_G_loss).mean())
    losses_D.append(np.array(batch_D_loss).mean())

    # Display generated images during training (every 10 epochs)
    if (epoch + 1) % DISPLAY_PER_EPOCHS == 0:
        print(f'Epoch [{epoch+1}/{EPOCHS}] | Loss D: {losses_D[-1]:.4f} | Loss G: {losses_G[-1]:.4f}')
        generated_imgs = fake_data.view(-1, 8, 8).detach().numpy()
        for i in range(10):
            plt.subplot(1,10, i + 1)
            plt.imshow(generated_imgs[i], cmap='gray')
            plt.axis('off')
        plt.show()




## コード説明

### Genarator
```python
class Generator(nn.Module):
    def __init__(self, latent_dim, hidden_dim1, hidden_dim2, img_dim):
        ...
        self.fc = nn.Sequential(
            nn.Linear(latent_dim, hidden_dim1),   # 入力は潜在ベクトル
            nn.ReLU(),
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Linear(hidden_dim2, img_dim),      # 出力は画像の次元（64）
            nn.Tanh()                             # 出力は[-1, 1] に正規化
        )
````
入力: latent_dim=64のランダムノイズ（潜在変数）
出力: img_dim=64（8x8画像をflattenしたもの）

活性化関数にReLUを使用し、最後だけTanh（出力が -1〜1 になるように）。
Tanhは画像のピクセル値を -1〜1 に収めるため（元データも -1〜1 に正規化する。） 。
*** 可視化ライブラリが[-1,1]でも動作するため。***


![](fig/8200-gan-generator-diagram1.jpg)

### Discriminator

```python
class Discriminator(nn.Module):
    def __init__(self, img_dim, hidden_dim1, hidden_dim2):
        ...
        self.fc = nn.Sequential(
            nn.Linear(img_dim, hidden_dim1),
            nn.ReLU(),
            nn.Dropout(0.3),                      # 過学習防止
            nn.Linear(hidden_dim1, hidden_dim2),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(hidden_dim2, 1),            # 出力は1ノード（本物 or 偽物）
            nn.Sigmoid()                          # 出力は確率値（0〜1）
        )

```

入力: img_dim、画像サイズ64）
出力: スカラー、本物である確率（0〜1）

![](fig/8200-gan-discriminator-diagram1.jpg)

### 損失関数

```python
criterion = nn.BCELoss()
```

$$
L(y,\hat{y}) = - y \log(\hat{y}) - (1 - y)\log\bigl(1 - \hat{y}\bigr)
$$

---

![](fig/8200-gan-loss-functions.jpg)

### training loop

```python
# 損失関数とオプティマイザーを定義
criterion = nn.BCELoss()
optimizer_G = optim.Adam(generator.parameters(), lr=0.001)
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.001)

for epoch in range(EPOCHS):
    for i in range(0, X_train.size(0), BATCH_SIZE):
        ...
        # ディスクリミネーターのトレーニング
        optimizer_D.zero_grad()
        ...
        loss_D.backward()
        optimizer_D.step()

        # ジェネレーターのトレーニング
        optimizer_G.zero_grad()
        ...
        loss_G.backward()
        optimizer_G.step()
```

### 損失関数の変化


- D loss << G loss：Generatorの偽物画像を簡単に見抜いている
  
- D loss >> G loss：Discriminatorが「本物」と「偽物」の区別がつかなくなっている
などが良く起きる。どちらも妥当でない。

両者のLossは均衡させたい。


In [ ]:
# Plot the loss values
LINEAR_SCALE=True

plt.figure()
if LINEAR_SCALE:
    plt.plot(losses_G, label='Generator Loss (linear scale)')
    plt.plot(losses_D, label='Discriminator Loss (linear scale)')
else:
    plt.plot(np.log(losses_G), label='Generator Loss (log scale)')
    plt.plot(np.log(losses_D), label='Discriminator Loss (log scale)')
plt.legend()
plt.show()